# Lab 2 | ARIMA + AIC/BIC + chẩn đoán phần dư

**Mục bình chọn:** ARIMA — Dataset 1 và ARIMA baseline Dataset 2

**Hạn bàn giao:** 25/10/2026, 20:00 — bàn giao mô hình/code fit cho phần forecast và so sánh.

> Bạn xây dựng và kiểm tra mô hình ARIMA: thử cấu hình, so AIC/BIC, kiểm tra hội tụ và phần dư. Ngoài mô hình chính của Dataset 1, bạn phải có ARIMA trên Dataset 2 để phần nâng cao so sánh với SARIMA.

**Đối chiếu đề:** Bài cơ bản: Bài 5 mục chia dữ liệu, thiết lập ARIMA và chẩn đoán phần dư; câu hỏi 1–5. Nâng cao: ARIMA baseline trên chính Dataset 2.

Đây là mô tả nhận việc, không phải lời giải. Các cell code vẫn giữ nguyên `# TODO`.

## Đầu vào, phụ thuộc và việc làm song song

**Cần nhận:** d/D và phép xử lý từ ADF; các cấu hình ARIMA ứng viên từ ACF/PACF.
**Có thể chuẩn bị trước:** khung fit/so sánh mô hình, bảng AIC/BIC và khung đồ thị phần dư.
Khi có cấu hình, ARIMA và SARIMA làm song song; bạn không phải chờ SARIMA fit xong.

## Cụ thể cần làm gì?

1. Xác nhận train/test 80/20 theo thời gian: Dataset 1 là 163/41, Dataset 2 là 115/29, không shuffle.
2. Fit 1–2 cấu hình ARIMA trên train1 gốc, ghi rõ (p,d,q), AIC, BIC và trạng thái hội tụ.
   Không đưa chuỗi đã sai phân vào rồi lại dùng d>0 để sai phân thêm lần nữa.
3. So sánh ứng viên bằng thông tin trong train và chẩn đoán; không dùng lỗi test để chọn cấu hình.
4. Vẽ phần dư theo thời gian và ACF phần dư; kiểm tra mean gần 0 và dấu hiệu tự tương quan.
   Nếu cần bổ sung kiểm định, ghi cách làm và diễn giải, không chỉ đưa bảng số.
5. Fit ARIMA baseline trên train2 để phần nâng cao có mô hình đối chiếu với SARIMA.
6. Trả lời câu hỏi 1–5 Bài 5: vì sao không shuffle, vai trò test, ý nghĩa p/d/q,
   hội tụ, white noise và hướng xử lý khi phần dư còn tự tương quan.

## Cần bàn giao những gì?

**Gửi cho phần forecast/đánh giá:**
- Code fit có thể chạy lại từ đầu, cấu hình cuối, thông tin dữ liệu/index và cách tạo forecast.
- Bảng AIC/BIC/hội tụ, đồ thị phần dư và kết luận chẩn đoán cho Dataset 1.
- ARIMA baseline Dataset 2 cùng cấu hình và code tái tạo; không bỏ sót phần này.
- Mốc test/horizon phải giữ nguyên: 41 tháng Dataset 1, 29 tháng Dataset 2.
- Không chỉ gửi tên biến fitted model trong kernel hoặc một file pickle phụ thuộc môi trường.
  Phần forecast chịu trách nhiệm tổng hợp đồ thị/metrics cuối; bạn không cần làm trùng toàn bộ bảng đánh giá.

## Quy ước chung

- Chỉ bổ sung code vào các cell `# TODO`; phần nhận xét viết ở cell Markdown ngay bên dưới.
- Đọc CSV chung trong `data/`; không sửa nguồn, không tạo một bản dataset khác.
- Dataset 1 (`a10.csv`) dùng `date`/`value`; Dataset 2 (`airline-passengers.csv`) dùng `Month`/`Passengers`.
- Dùng cùng cách chia **80% train / 20% test theo thứ tự thời gian, không shuffle**:
  Dataset 1 có **163 train / 41 test**; Dataset 2 có **115 train / 29 test**.
- Có thể dùng helper `src/lab2_data.py`, nhưng phần import/load vẫn để bạn tự viết.
  Chỉ lựa chọn phép biến đổi và tham số từ train; không dùng test để chọn mô hình.
- Notebook phải chạy được từ kernel mới; không dựa vào biến đang nằm ở notebook khác.
- Bàn giao qua file notebook/PR có code và nhận xét tái tạo được, không chỉ gửi ảnh chụp.

## Khi nào xem là hoàn thành?

- [ ] Dataset 1 có bảng so sánh mô hình, hội tụ và chẩn đoán phần dư.
- [ ] Có ARIMA baseline Dataset 2, không chỉ làm Dataset 1.
- [ ] Code fit và cấu hình bàn giao đủ để tái tạo forecast.
- [ ] Trả lời câu hỏi 1–5 Bài 5.

---
## Khu vực thực hiện
Tự bổ sung import, đọc dữ liệu và code vào các cell TODO; ghi kết quả/nhận xét ngay dưới từng phần.

### 1. Fit và so sánh ARIMA trên Dataset 1

Nhận các cấu hình từ ACF/PACF; fit trên train1 gốc.

In [ ]:
import pandas as pd
import warnings
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Đọc Dataset 1
df1 = pd.read_csv("data/a10.csv", parse_dates=["date"])
df1 = df1.sort_values("date").set_index("date")

y1 = df1["value"].astype(float).asfreq("MS")

# Chia dữ liệu theo thời gian, không shuffle
split1 = int(len(y1) * 0.8)
train1 = y1.iloc[:split1]
test1 = y1.iloc[split1:]

print("Số quan sát train1:", len(train1))
print("Số quan sát test1:", len(test1))

# Các cấu hình thử nghiệm, cần đối chiếu với ACF/PACF và sai phân
orders1 = [(1, 1, 1), (2, 1, 1)]

models1 = {}
results1 = []

for order in orders1:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fitted = ARIMA(train1, order=order).fit()

        converged = fitted.mle_retvals.get("converged", None)

        warning_text = "; ".join(str(w.message) for w in caught)

        models1[order] = fitted

        results1.append({
            "Model": f"ARIMA{order}",
            "p": order[0],
            "d": order[1],
            "q": order[2],
            "AIC": fitted.aic,
            "BIC": fitted.bic,
            "Converged": converged,
            "Warning": warning_text
        })

    except Exception as e:
        results1.append({
            "Model": f"ARIMA{order}",
            "p": order[0],
            "d": order[1],
            "q": order[2],
            "AIC": None,
            "BIC": None,
            "Converged": False,
            "Warning": str(e)
        })

comparison1 = pd.DataFrame(results1)

print("\nBảng so sánh mô hình Dataset 1:")
print(comparison1.to_string(index=False))

**Nhận xét:** Hai mô hình ARIMA được thử nghiệm trên tập train của Dataset 1 là ARIMA(1,1,1) và ARIMA(2,1,1). Các mô hình được so sánh dựa trên AIC, BIC và trạng thái hội tụ. Mô hình có AIC/BIC thấp hơn thường được ưu tiên khi so sánh các mô hình phù hợp trên cùng tập dữ liệu. Tuy nhiên, việc lựa chọn cuối cùng còn phụ thuộc vào chẩn đoán phần dư. Các cấu hình thử nghiệm cần được đối chiếu với kết quả ACF/PACF do chưa đánh giá trên dữ liệu thực tế, chưa thể kết luận mô hình nào tốt nhất.

### 2. Chẩn đoán phần dư và giải thích

So sánh AIC/BIC, hội tụ và chẩn đoán phần dư.

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

# Chỉ xét các mô hình có AIC hợp lệ
valid_models1 = {
    order: model
    for order, model in models1.items()
    if pd.notna(model.aic)
}

if not valid_models1:
    raise ValueError("Không có mô hình Dataset 1 fit thành công.")

# Chọn ứng viên có AIC thấp nhất
best_order1 = min(
    valid_models1,
    key=lambda order: valid_models1[order].aic
)

best_model1 = valid_models1[best_order1]
residuals1 = best_model1.resid.dropna()

print("Mô hình được chọn để chẩn đoán:", best_order1)
print("AIC:", best_model1.aic)
print("BIC:", best_model1.bic)
print("Trạng thái hội tụ:",
      best_model1.mle_retvals.get("converged", None))
print("Trung bình phần dư:", residuals1.mean())

# 1. Phần dư theo thời gian
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(residuals1)
ax.axhline(0, linestyle="--")
ax.set_title(f"Residuals - ARIMA{best_order1}")
ax.set_xlabel("Thời gian")
ax.set_ylabel("Phần dư")
plt.tight_layout()
plt.show()

# 2. ACF phần dư
plot_acf(residuals1, lags=24)
plt.title("ACF của phần dư - Dataset 1")
plt.tight_layout()
plt.show()

# 3. Kiểm định Ljung-Box
lb1 = acorr_ljungbox(
    residuals1,
    lags=[12, 24],
    return_df=True
)

print("\nKết quả kiểm định Ljung-Box:")
print(lb1)

**Nhận xét:**

- **AIC/BIC:** Giá trị thấp hơn thường thể hiện sự cân bằng tốt hơn giữa mức độ phù hợp và độ phức tạp của mô hình khi so sánh trên cùng dữ liệu.

- **Hội tụ:** Nếu `Converged = True`, thuật toán ước lượng đã báo hội tụ. Nếu là `False` hoặc có cảnh báo, cần kiểm tra lại cấu hình và kết quả ước lượng.

- **Trung bình phần dư:** Giá trị trung bình của phần dư kỳ vọng gần 0. Tuy nhiên, điều này không chứng minh phần dư hoàn toàn ngẫu nhiên.

- **ACF phần dư:** Nếu còn nhiều cột vượt khỏi khoảng tin cậy, mô hình có thể chưa giải thích hết cấu trúc của chuỗi thời gian.

- **Ljung–Box:** Nếu `lb_pvalue > 0.05`, chưa có đủ bằng chứng để kết luận phần dư còn tự tương quan tại các độ trễ được kiểm tra. Nếu `lb_pvalue <= 0.05`, có dấu hiệu tự tương quan còn lại.

**Hạn chế:**

Việc chọn mô hình theo AIC thấp nhất chưa đảm bảo mô hình có khả năng dự báo tốt nhất. Nếu phần dư còn tự tương quan, cần thử điều chỉnh các tham số `p`, `q`, kiểm tra lại bậc sai phân `d` hoặc xem xét mô hình có thành phần mùa vụ như SARIMA.

### 3. ARIMA baseline trên Dataset 2

Fit ARIMA baseline trên train2 cho so sánh với SARIMA.

In [ ]:
# Đọc Dataset 2
df2 = pd.read_csv(
    "data/airline-passengers.csv",
    parse_dates=["Month"]
)

df2 = df2.sort_values("Month").set_index("Month")

y2 = df2["Passengers"].astype(float).asfreq("MS")

# Chia train/test theo thời gian
split2 = int(len(y2) * 0.8)

train2 = y2.iloc[:split2]
test2 = y2.iloc[split2:]

print("Số quan sát train2:", len(train2))
print("Số quan sát test2:", len(test2))

# Thử hai cấu hình ARIMA baseline
orders2 = [(1, 1, 1), (2, 1, 1)]

models2 = {}
results2 = []

for order in orders2:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fitted = ARIMA(
                train2,
                order=order
            ).fit()

        models2[order] = fitted

        results2.append({
            "Model": f"ARIMA{order}",
            "AIC": fitted.aic,
            "BIC": fitted.bic,
            "Converged": fitted.mle_retvals.get(
                "converged", None
            ),
            "Warning": "; ".join(
                str(w.message) for w in caught
            )
        })

    except Exception as e:
        results2.append({
            "Model": f"ARIMA{order}",
            "AIC": None,
            "BIC": None,
            "Converged": False,
            "Warning": str(e)
        })

comparison2 = pd.DataFrame(results2)

print("\nBảng so sánh ARIMA baseline Dataset 2:")
print(comparison2.to_string(index=False))

**Nhận xét:**

Dataset 2 sử dụng dữ liệu số hành khách hàng không theo tháng từ năm 1949 đến năm 1960. Dữ liệu được chia thành tập huấn luyện (train) gồm 115 quan sát và tập kiểm tra (test) gồm 29 quan sát theo đúng thứ tự thời gian

Hai cấu hình ARIMA(1,1,1) và ARIMA(2,1,1) được thử nghiệm nhằm lựa chọn mô hình baseline dựa trên AIC, BIC, trạng thái hội tụ và chẩn đoán phần dư

**Hạn chế:**

Chuỗi số hành khách có xu hướng tăng và đặc điểm mùa vụ theo chu kỳ 12 tháng. Mô hình ARIMA thông thường có thể chưa mô tả đầy đủ yếu tố mùa vụ, do đó kết quả dự báo có thể chưa sát với dữ liệu thực tế. Vì vậy, mô hình ARIMA baseline sẽ được sử dụng để so sánh với mô hình SARIMA trong phần nâng cao. Việc lựa chọn mô hình cuối cùng cần dựa trên kết quả thực nghiệm và chẩn đoán phần dư.

### 4. Bàn giao mô hình và cách dự báo

Bàn giao cấu hình, code fit và dự báo theo đúng index test.

In [ ]:
# Chọn mô hình ARIMA Dataset 1 đã fit thành công
best_order1 = min(
    models1,
    key=lambda order: models1[order].aic
)
best_model1 = models1[best_order1]

# Chọn mô hình ARIMA baseline Dataset 2
best_order2 = min(
    models2,
    key=lambda order: models2[order].aic
)
best_model2 = models2[best_order2]

# Dự báo đúng số tháng của tập test
forecast1 = best_model1.forecast(steps=len(test1))
forecast1.index = test1.index

forecast2 = best_model2.forecast(steps=len(test2))
forecast2.index = test2.index

# Bảng kết quả dự báo Dataset 1
forecast_table1 = pd.DataFrame({
    "Actual": test1,
    "Forecast": forecast1
})

print("Dataset 1:")
print("Order:", best_order1)
print(forecast_table1.head())

# Bảng kết quả dự báo Dataset 2
forecast_table2 = pd.DataFrame({
    "Actual": test2,
    "Forecast": forecast2
})

print("\nDataset 2:")
print("Order:", best_order2)
print(forecast_table2.head())

# Vẽ dự báo Dataset 1
plt.figure(figsize=(12, 5))
plt.plot(train1, label="Train")
plt.plot(test1, label="Test - Actual")
plt.plot(forecast1, label="ARIMA Forecast")
plt.title(f"ARIMA{best_order1} - Dataset 1")
plt.xlabel("Thời gian")
plt.ylabel("Doanh số")
plt.legend()
plt.tight_layout()
plt.show()

# Vẽ dự báo Dataset 2
plt.figure(figsize=(12, 5))
plt.plot(train2, label="Train")
plt.plot(test2, label="Test - Actual")
plt.plot(forecast2, label="ARIMA Forecast")
plt.title(f"ARIMA{best_order2} - Dataset 2")
plt.xlabel("Thời gian")
plt.ylabel("Số hành khách")
plt.legend()
plt.tight_layout()
plt.show()

**Nhận xét:**

Sau khi hoàn thành quá trình huấn luyện, mô hình ARIMA được sử dụng để dự báo trên tập test của từng bộ dữ liệu. Dataset 1 có 41 quan sát test và Dataset 2 có 29 quan sát test. Kết quả dự báo được gắn với index thời gian tương ứng để thuận tiện cho việc so sánh giữa giá trị thực tế và giá trị dự báo.

**Hạn chế:**

Kết quả dự báo phụ thuộc vào cấu hình ARIMA được lựa chọn và đặc điểm của từng chuỗi thời gian. Đối với Dataset 2, mô hình ARIMA có thể chưa mô tả tốt yếu tố mùa vụ theo chu kỳ 12 tháng. Vì vậy, cần tiếp tục so sánh với SARIMA để đánh giá khả năng dự báo. Ngoài ra, việc lựa chọn mô hình không nên chỉ dựa trên AIC/BIC mà cần kết hợp kết quả chẩn đoán phần dư và các chỉ số đánh giá trên tập test.